## Stochastic Variational Inference (SVI)

Stochastic Variational Inference (SVI) is a Bayesian modeling framework that approximates the intractable posterior distribution $p(R | \mathcal{D})$ of the readout weights by introducing a simpler, parameterized distribution $q_\phi(R)$, known as the **variational distribution**. 

The goal is to find the parameters $\phi$ that make $q_\phi(R)$ as close as possible to the true posterior. This is achieved by minimizing the **Kullback-Leibler (KL) divergence** between the two distributions:

$$
\min_\phi \text{KL}(q_\phi(R) || p(R | \mathcal{D}))
$$

Since the true posterior is unknown, this minimization is equivalent to maximizing the **Evidence Lower Bound (ELBO)**. The ELBO loss function is defined as:

$$
\text{ELBO}(\phi) = \mathbb{E}_{q_\phi(R)}[\log p(\mathcal{D} | R)] - \text{KL}(q_\phi(R) || p(R))
$$

Where:
1.  **Likelihood Term** ($\mathbb{E}_{q_\phi(R)}[\log p(\mathcal{D} | R)]$): Measures how well the model fits the data given the reservoir states.
2.  **Complexity/Regularization Term** ($\text{KL}(q_\phi(R) || p(R))$): Penalizes the variational distribution for deviating too much from the prior $p(R)$, acting as a natural regularizer.

SVI leverages **stochastic optimization** to maximize the ELBO, making it significantly more scalable to large datasets than traditional MCMC methods. In our project, we use SVI to obtain a probabilistic readout $R$ that provides not just a prediction, but a full distribution of possible outcomes, allowing also for **uncertainty quantification**.

### Prior and Likelihood Specifications

To implement SVI in our ESN, we define the probabilistic components as follows:

1.  **Prior $p(R)$**: We assume a **Gaussian Prior** for the readout weights, $R \sim \mathcal{N}(0, \sigma^2 I)$. This acts as a ridge regularization (L2), preventing the weights from exploding and helping the model to generalize.

2.  **Variational Distribution $q_\phi(R)$**: To approximate the posterior, we use a **LowRank Multivariate Gaussian** distribution. 
    * **Why LowRank:** A full covariance matrix for a large reservoir ($N$) would have $N^2$ parameters, making optimization computationally prohibitive. 
    * **The Structure**: By using a LowRank parametrization, we represent the covariance as $\Sigma = D + VV^T$ (where $D$ is diagonal and $V$ is a low-rank matrix). This captures the main correlations between reservoir states with a fraction of the parameters, balancing expressive power and efficiency.

3.  **Likelihood $p(\mathcal{D} | R)$**: We assume a **Gaussian Likelihood** (for regression tasks), where the observed output $y_t$ is distributed around the model prediction $g(Rs_t)$ with a certain noise $\epsilon$:
    $$p(y_t | s_t, R) = \mathcal{N}(y_t | Rs_t, \sigma_{obs}^2)$$